# Memory-Efficient Training (For Limited Hardware)
This notebook trains on a reasonable sample size that fits your hardware constraints.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## 1. Load Optimized Dataset Size

In [3]:
# Load full dataset first to check size
df = pd.read_csv('train.csv')
print(f"Full dataset: {df.shape[0]:,} rows")

# MEMORY-SAFE APPROACH: Use 100K sample (25% of data)
# This gives good performance while fitting your 16GB RAM
df_sample = df.sample(100000, random_state=42)

print(f"Using sample: {len(df_sample):,} rows ({len(df_sample)/len(df):.1%} of full data)")
print(f"Sample class distribution: {df_sample['is_duplicate'].mean():.2%} duplicates")
print(f"Original class distribution: {df['is_duplicate'].mean():.2%} duplicates")

Full dataset: 404,290 rows
Using sample: 100,000 rows (24.7% of full data)
Sample class distribution: 37.12% duplicates
Original class distribution: 36.92% duplicates


## 2. Memory-Efficient Feature Creation

In [4]:
# Import helper functions
import sys
sys.path.append('.')
from helper import preprocess, test_fetch_token_features, test_fetch_length_features, test_fetch_fuzzy_features
from helper import test_common_words, test_total_words

def create_features_batch(df_subset, batch_size=5000):
    """Create features in batches to save memory"""
    features = []
    target = []
    
    for start_idx in range(0, len(df_subset), batch_size):
        end_idx = min(start_idx + batch_size, len(df_subset))
        batch_df = df_subset.iloc[start_idx:end_idx]
        
        print(f"Processing batch {start_idx//batch_size + 1}/{(len(df_subset)-1)//batch_size + 1} ({start_idx}-{end_idx})")
        
        batch_features = []
        batch_target = []
        
        for idx, row in batch_df.iterrows():
            q1 = str(row['question1'])
            q2 = str(row['question2'])
            
            # Preprocess
            q1 = preprocess(q1)
            q2 = preprocess(q2)
            
            # Basic features
            feature_list = []
            feature_list.append(len(q1))
            feature_list.append(len(q2))
            feature_list.append(len(q1.split(" ")))
            feature_list.append(len(q2.split(" ")))
            feature_list.append(test_common_words(q1, q2))
            feature_list.append(test_total_words(q1, q2))
            feature_list.append(round(test_common_words(q1, q2) / (test_total_words(q1, q2) + 0.0001), 2))
            
            # Token features
            feature_list.extend(test_fetch_token_features(q1, q2))
            
            # Length features
            feature_list.extend(test_fetch_length_features(q1, q2))
            
            # Fuzzy features
            feature_list.extend(test_fetch_fuzzy_features(q1, q2))
            
            batch_features.append(feature_list)
            batch_target.append(row['is_duplicate'])
        
        features.extend(batch_features)
        target.extend(batch_target)
        
        # Clear batch memory
        del batch_df, batch_features, batch_target
    
    return np.array(features), np.array(target)

print("Batch feature creation function ready")

Batch feature creation function ready


In [5]:
# Create features with memory management
print(f"Creating features for {len(df_sample)} rows (batch processing)...")
X, y = create_features_batch(df_sample, batch_size=5000)
print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Memory usage: ~{X.nbytes / 1024 / 1024:.1f} MB for features")

Creating features for 100000 rows (batch processing)...
Processing batch 1/20 (0-5000)
Processing batch 2/20 (5000-10000)
Processing batch 3/20 (10000-15000)
Processing batch 4/20 (15000-20000)
Processing batch 5/20 (20000-25000)
Processing batch 6/20 (25000-30000)
Processing batch 7/20 (30000-35000)
Processing batch 8/20 (35000-40000)
Processing batch 9/20 (40000-45000)
Processing batch 10/20 (45000-50000)
Processing batch 11/20 (50000-55000)
Processing batch 12/20 (55000-60000)
Processing batch 13/20 (60000-65000)
Processing batch 14/20 (65000-70000)
Processing batch 15/20 (70000-75000)
Processing batch 16/20 (75000-80000)
Processing batch 17/20 (80000-85000)
Processing batch 18/20 (85000-90000)
Processing batch 19/20 (90000-95000)
Processing batch 20/20 (95000-100000)

Features shape: (100000, 22)
Target shape: (100000,)
Memory usage: ~16.8 MB for features


## 3. Train-Test Split & Scaling

In [6]:
# Split data: 70% train, 30% test with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Clear original arrays to save memory
del X, X_train, X_test

print("Features scaled and memory optimized")

Training set: 70000 samples
Test set: 30000 samples
Features scaled and memory optimized


## 4. Model Training with Cross-Validation

In [7]:
# Use optimized Gradient Boosting for your hardware
print("Training Gradient Boosting model...")
model = GradientBoostingClassifier(
    n_estimators=100,      # Reduced for memory
    learning_rate=0.1,     # Slightly higher for faster convergence
    max_depth=6,           # Reduced depth
    min_samples_split=20,  # Higher to reduce overfitting
    subsample=0.8,
    random_state=42,
    verbose=1              # Show progress
)

# Quick cross-validation (3-fold instead of 5)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='f1')

print(f"\nCross-validation F1 scores: {cv_scores}")
print(f"Mean CV F1 score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

Training Gradient Boosting model...
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           1.2643           0.0543           18.00s
         2           1.2167           0.0409           16.82s
         3           1.1776           0.0392           15.97s
         4           1.1447           0.0327           15.38s
         5           1.1165           0.0293           14.82s
         6           1.0920           0.0216           14.43s
         7           1.0702           0.0186           14.10s
         8           1.0516           0.0179           13.84s
         9           1.0381           0.0246           13.57s
        10           1.0213           0.0054           13.36s
        20           0.9419           0.0158           11.64s
        30           0.9036          -0.0026           10.04s
        40           0.8848          -0.0101            8.55s
        50           0.8766          -0.0053            7.12s
        60           0.8601      

In [8]:
# Train final model
print("\nTraining final model on full training set...")
model.fit(X_train_scaled, y_train)
print("Model training completed!")


Training final model on full training set...
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           1.2633           0.0541           25.83s
         2           1.2189           0.0520           23.77s
         3           1.1789           0.0302           23.11s
         4           1.1459           0.0311           22.78s
         5           1.1195           0.0345           22.11s
         6           1.0960           0.0249           21.58s
         7           1.0740           0.0147           21.17s
         8           1.0582           0.0263           21.04s
         9           1.0427           0.0166           20.74s
        10           1.0266           0.0035           20.52s
        20           0.9464           0.0069           17.72s
        30           0.9150          -0.0017           15.29s
        40           0.8999           0.0024           13.04s
        50           0.8848          -0.0096           10.84s
        60           0.

## 5. Model Evaluation

In [ ]:
# Predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)
y_test_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Evaluation metrics
print("="*60)
print("MODEL EVALUATION METRICS (100K Sample)")
print("="*60)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1 Score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_auc:.4f}")
print(f"\nCross-validation F1: {cv_scores.mean():.4f}")

MODEL EVALUATION METRICS (100K Sample)
Accuracy:  0.7576
Precision: 0.6655
Recall:    0.6978
F1 Score:  0.6813
ROC-AUC:   0.8449

Cross-validation F1: 0.6802


In [13]:
confusion_matrix(y_test, y_test_pred)

array([[14958,  3906],
       [ 3365,  7771]])

## 6. Save Model

In [ ]:
with open('model_100k_sample.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('scaler_100k.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model and scaler saved!")
print("Files: 'model_100k_sample.pkl', 'scaler_100k.pkl'")

Model and scaler saved!
Files: 'model_100k_sample.pkl', 'scaler_100k.pkl'
